# Tema 9 - Diseño de Controladores en el Dominio de la Frecuencia

**Fundamentos de Control - GIERM**

---

## Objetivos de aprendizaje

- Aplicar el **criterio de Nyquist** para determinar la estabilidad de un sistema realimentado
- Calcular e interpretar los **márgenes de ganancia y fase** a partir de diagramas de Bode y Nyquist
- Diseñar **compensadores en adelanto** (lead) para mejorar el margen de fase
- Diseñar **compensadores en atraso** (lag) para mejorar el margen de ganancia
- Analizar el efecto de controladores P, PI, PD y PID en el dominio frecuencial
- Relacionar especificaciones frecuenciales con especificaciones temporales

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import signal

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

COLOR_PRINCIPAL = '#2171b5'   # azul - curvas principales
COLOR_RECTA = '#cb181d'       # rojo - límites, compensador
COLOR_PUNTO = '#238b45'       # verde - puntos clave, resultados
COLOR_COMP = '#ff7f00'        # naranja - compensador / despues
COLOR_FASE = '#6a3d9a'        # morado - curvas de fase

print('Configuración lista.')

---

## 1. Criterio de Nyquist

El **criterio de Nyquist** permite determinar la estabilidad de un sistema en lazo cerrado a partir de la respuesta frecuencial en lazo abierto $G(j\omega)H(j\omega)$.

### 1.1 Enunciado del criterio

Sea $G_{BA}(s) = G(s)H(s)$ la función de transferencia en **bucle abierto** con $P$ polos en el semiplano derecho (SPD). El número de polos inestables del sistema en **lazo cerrado** es:

$$\boxed{Z = N + P}$$

donde:
- $Z$ = número de ceros de $1 + G_{BA}(s)$ en el SPD (polos inestables en lazo cerrado)
- $N$ = número de **encirculamientos** del punto $(-1, 0)$ en sentido horario
- $P$ = número de polos de $G_{BA}(s)$ en el SPD

**Para estabilidad en lazo cerrado:** $Z = 0 \implies N = -P$

- Si $G_{BA}(s)$ es **estable en lazo abierto** ($P = 0$): el sistema en lazo cerrado es estable si y solo si $N = 0$ (no hay encirculamientos de $(-1, 0)$)
- Si $G_{BA}(s)$ tiene polos en el SPD ($P > 0$): necesitamos $N = -P$ encirculamientos en sentido **antihorario**

### 1.2 Conteo de encirculamientos

Para contar $N$, se traza un vector desde $(-1, 0)$ hasta un punto del diagrama de Nyquist y se sigue la curva completa:
- Cada vuelta completa **horaria** suma $+1$ a $N$
- Cada vuelta completa **antihoraria** suma $-1$ a $N$

**Truco para el examen:** Si la curva de Nyquist deja el punto $(-1, 0)$ a su **derecha** al recorrerla en el sentido de $\omega$ creciente, hay un encirculamiento horario.

In [ ]:
# Diagrama de Nyquist: sistema estable vs inestable
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Sistema estable: G(s) = 1 / (s+1)(s+2)(s+3) ---
num_est = [1.0]
den_est = np.polymul(np.polymul([1, 1], [1, 2]), [1, 3])
w = np.logspace(-2, 2, 2000)
s_jw = 1j * w

G_est = np.polyval(num_est, s_jw) / np.polyval(den_est, s_jw)

ax = axes[0]
ax.plot(G_est.real, G_est.imag, color=COLOR_PRINCIPAL, lw=2.5, label=r'$\omega > 0$')
ax.plot(G_est.real, -G_est.imag, color=COLOR_PRINCIPAL, lw=2.5, ls='--', alpha=0.5, label=r'$\omega < 0$')
ax.plot(-1, 0, 'x', color=COLOR_RECTA, ms=14, mew=3, zorder=5, label=r'$(-1, 0)$')
ax.set_xlabel(r'Parte Real')
ax.set_ylabel(r'Parte Imaginaria')
ax.set_title(r'Sistema ESTABLE: $G(s) = \frac{1}{(s+1)(s+2)(s+3)}$')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.3, 0.25)
ax.set_ylim(-0.2, 0.2)
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.annotate(r'$N = 0 \to Z = 0$' + '\nEstable en LC',
            xy=(-1, 0), xytext=(-0.25, 0.12),
            fontsize=12, color=COLOR_PUNTO, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2),
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_PUNTO))

# --- Sistema inestable: G(s) = 10 / (s+1)(s+2)(s+3) ---
num_inst = [10.0]
G_inst = np.polyval(num_inst, s_jw) / np.polyval(den_est, s_jw)

ax = axes[1]
ax.plot(G_inst.real, G_inst.imag, color=COLOR_RECTA, lw=2.5, label=r'$\omega > 0$')
ax.plot(G_inst.real, -G_inst.imag, color=COLOR_RECTA, lw=2.5, ls='--', alpha=0.5, label=r'$\omega < 0$')
ax.plot(-1, 0, 'x', color='black', ms=14, mew=3, zorder=5, label=r'$(-1, 0)$')
ax.set_xlabel(r'Parte Real')
ax.set_ylabel(r'Parte Imaginaria')
ax.set_title(r'Sistema INESTABLE: $G(s) = \frac{10}{(s+1)(s+2)(s+3)}$')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-2.5, 2.0)
ax.set_ylim(-2.0, 2.0)
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.annotate(r'$(-1,0)$ encerrado' + '\n' + r'$N=1 \to Z=1$' + '\nInestable en LC',
            xy=(-1, 0), xytext=(0.5, 1.2),
            fontsize=12, color=COLOR_RECTA, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_RECTA, lw=2),
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_RECTA))

plt.tight_layout()
plt.show()

---

## 2. Criterio de Nyquist reducido (sistemas de fase mínima)

Para **sistemas de fase mínima** (sin polos ni ceros en el SPD, sin retardos puros), el criterio se simplifica:

$$\boxed{\text{Estable en LC} \iff (-1, 0) \text{ queda a la IZQUIERDA de la curva de Nyquist}}$$

Esto equivale a decir que la curva de Nyquist **no encierra** el punto $(-1, 0)$.

**Procedimiento práctico:**
1. Trazar solo la parte $\omega \geq 0$ del diagrama de Nyquist
2. Observar dónde la curva cruza el eje real negativo
3. Si el cruce ocurre **antes** de $-1$ (más a la derecha), el sistema es estable
4. Si el cruce ocurre **después** de $-1$ (más a la izquierda), el sistema es inestable

**Error frecuente:** Este criterio reducido NO aplica si hay polos en el SPD o retardos puros. En esos casos hay que usar el criterio completo.

---

## 3. Márgenes de estabilidad

Los márgenes de estabilidad cuantifican **cuánto margen** tiene un sistema antes de volverse inestable.

### 3.1 Margen de ganancia ($MG$)

Se evalúa en la **frecuencia de cruce de fase** $\omega_{pc}$, donde la fase vale $-180°$:

$$\boxed{MG = -20\log_{10}|G(j\omega_{pc})| \;\text{dB}}$$

- Si $MG > 0$ dB $\to$ el sistema es estable (la ganancia puede aumentar $MG$ dB antes de inestabilidad)
- Si $MG < 0$ dB $\to$ el sistema es inestable

### 3.2 Margen de fase ($MF$)

Se evalúa en la **frecuencia de cruce de ganancia** $\omega_{gc}$, donde $|G(j\omega_{gc})| = 1$ (0 dB):

$$\boxed{MF = 180° + \angle G(j\omega_{gc})}$$

- Si $MF > 0°$ $\to$ el sistema es estable (la fase puede disminuir $MF$ grados antes de inestabilidad)
- Si $MF < 0°$ $\to$ el sistema es inestable

### 3.3 Valores típicos recomendados

| Especificación | Valor típico | Notas |
|----------------|-------------|-------|
| $MG$ | $> 6$ dB (mejor $> 12$ dB) | Margen frente a variaciones de ganancia |
| $MF$ | $30° - 60°$ (ideal $\approx 45°$) | Relacionado con el amortiguamiento |

**Relación con amortiguamiento:** Para sistemas de segundo orden, $MF \approx 100\zeta$ para $\zeta < 0.7$.

In [ ]:
# Diagrama de Bode con márgenes MG y MF marcados
num = [2.0]
den = np.polymul(np.polymul([1, 1], [1, 0.5]), [1, 2])
sys_ba = signal.TransferFunction(num, den)

w_bode = np.logspace(-2, 2, 1000)
w_out, mag_db, phase_deg = signal.bode(sys_ba, w_bode)

# Encontrar frecuencia de cruce de ganancia (|G| = 0 dB)
idx_gc = np.argmin(np.abs(mag_db))
w_gc = w_out[idx_gc]
phase_gc = phase_deg[idx_gc]
MF = 180 + phase_gc

# Encontrar frecuencia de cruce de fase (fase = -180)
idx_pc = np.argmin(np.abs(phase_deg + 180))
w_pc = w_out[idx_pc]
mag_pc = mag_db[idx_pc]
MG = -mag_pc

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# --- Magnitud ---
ax1.semilogx(w_out, mag_db, color=COLOR_PRINCIPAL, lw=2.5)
ax1.axhline(0, color='gray', ls='--', lw=1)
ax1.axvline(w_gc, color=COLOR_PUNTO, ls=':', lw=1.5, alpha=0.7)
ax1.axvline(w_pc, color=COLOR_RECTA, ls=':', lw=1.5, alpha=0.7)

# Marcar MG en magnitud
ax1.annotate('', xy=(w_pc, 0), xytext=(w_pc, mag_pc),
             arrowprops=dict(arrowstyle='<->', color=COLOR_RECTA, lw=2.5))
ax1.text(w_pc * 1.3, mag_pc / 2, f'MG = {MG:.1f} dB',
         fontsize=13, color=COLOR_RECTA, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_RECTA))

ax1.plot(w_gc, 0, 'o', color=COLOR_PUNTO, ms=10, zorder=5)
ax1.plot(w_pc, mag_pc, 'o', color=COLOR_RECTA, ms=10, zorder=5)
ax1.set_ylabel(r'Magnitud (dB)')
ax1.set_title(r'Diagrama de Bode con márgenes - $G(s) = \frac{2}{(s+0.5)(s+1)(s+2)}$')
ax1.grid(True, alpha=0.3, which='both')
ax1.legend([r'$|G(j\omega)|$'], fontsize=11, loc='upper right')

# --- Fase ---
ax2.semilogx(w_out, phase_deg, color=COLOR_FASE, lw=2.5)
ax2.axhline(-180, color='gray', ls='--', lw=1)
ax2.axvline(w_gc, color=COLOR_PUNTO, ls=':', lw=1.5, alpha=0.7)
ax2.axvline(w_pc, color=COLOR_RECTA, ls=':', lw=1.5, alpha=0.7)

# Marcar MF en fase
ax2.annotate('', xy=(w_gc, -180), xytext=(w_gc, phase_gc),
             arrowprops=dict(arrowstyle='<->', color=COLOR_PUNTO, lw=2.5))
ax2.text(w_gc * 1.5, phase_gc - 20, f'MF = {MF:.1f}\u00b0',
         fontsize=13, color=COLOR_PUNTO, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_PUNTO))

ax2.plot(w_gc, phase_gc, 'o', color=COLOR_PUNTO, ms=10, zorder=5)
ax2.plot(w_pc, -180, 'o', color=COLOR_RECTA, ms=10, zorder=5)

# Etiquetas de frecuencias
ax2.text(w_gc, -195, r'$\omega_{gc}$', fontsize=12, ha='center', color=COLOR_PUNTO, fontweight='bold')
ax2.text(w_pc, -195, r'$\omega_{pc}$', fontsize=12, ha='center', color=COLOR_RECTA, fontweight='bold')

ax2.set_xlabel(r'Frecuencia $\omega$ (rad/s)')
ax2.set_ylabel(r'Fase (grados)')
ax2.grid(True, alpha=0.3, which='both')
ax2.legend([r'$\angle G(j\omega)$'], fontsize=11, loc='upper right')

plt.tight_layout()
plt.show()

print(f'Frecuencia de cruce de ganancia: \u03c9_gc = {w_gc:.3f} rad/s')
print(f'Margen de fase: MF = {MF:.1f}\u00b0')
print(f'Frecuencia de cruce de fase: \u03c9_pc = {w_pc:.3f} rad/s')
print(f'Margen de ganancia: MG = {MG:.1f} dB')

In [ ]:
# Márgenes MG y MF en el diagrama de Nyquist
fig, ax = plt.subplots(figsize=(10, 9))

G_val = np.polyval(num, 1j * w_bode) / np.polyval(den, 1j * w_bode)

ax.plot(G_val.real, G_val.imag, color=COLOR_PRINCIPAL, lw=2.5, label=r'$G(j\omega),\;\omega > 0$')
ax.plot(G_val.real, -G_val.imag, color=COLOR_PRINCIPAL, lw=2.5, ls='--', alpha=0.5, label=r'$G(j\omega),\;\omega < 0$')
ax.plot(-1, 0, 'x', color='black', ms=14, mew=3, zorder=5)

# Punto en la frecuencia de cruce de fase
G_pc = np.polyval(num, 1j * w_pc) / np.polyval(den, 1j * w_pc)
ax.plot(G_pc.real, G_pc.imag, 'o', color=COLOR_RECTA, ms=12, zorder=5)

# MG: distancia desde G(jw_pc) hasta -1 sobre el eje real
ax.annotate('', xy=(-1, 0), xytext=(G_pc.real, 0),
            arrowprops=dict(arrowstyle='<->', color=COLOR_RECTA, lw=2.5))
ax.text((-1 + G_pc.real) / 2, 0.05, f'MG = {MG:.1f} dB',
        fontsize=12, color=COLOR_RECTA, fontweight='bold', ha='center',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_RECTA))

# Punto en la frecuencia de cruce de ganancia
G_gc = np.polyval(num, 1j * w_gc) / np.polyval(den, 1j * w_gc)
ax.plot(G_gc.real, G_gc.imag, 'o', color=COLOR_PUNTO, ms=12, zorder=5)

# MF: ángulo desde -180 hasta la fase real
theta_gc = np.angle(G_gc)
theta_arr = np.linspace(-np.pi, theta_gc, 50)
r_arc = 0.3
ax.plot(r_arc * np.cos(theta_arr), r_arc * np.sin(theta_arr), color=COLOR_PUNTO, lw=2)
ax.annotate(f'MF = {MF:.1f}\u00b0', xy=(r_arc * np.cos((theta_gc - np.pi) / 2), r_arc * np.sin((theta_gc - np.pi) / 2)),
            xytext=(0.1, -0.5),
            fontsize=12, color=COLOR_PUNTO, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2),
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_PUNTO))

# Círculo unitario de referencia
theta_circ = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta_circ), np.sin(theta_circ), 'k--', lw=0.8, alpha=0.3)

ax.set_xlabel(r'Parte Real')
ax.set_ylabel(r'Parte Imaginaria')
ax.set_title(r'Márgenes en el diagrama de Nyquist')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.0)

plt.tight_layout()
plt.show()

---

## 4. Especificaciones frecuenciales y su relación con el dominio temporal

### 4.1 Ancho de banda ($\omega_{BW}$)

El **ancho de banda** es la frecuencia a la cual la magnitud de la función de transferencia en lazo cerrado cae 3 dB respecto a su valor en $\omega = 0$:

$$\boxed{|T(j\omega_{BW})| = |T(0)| - 3\;\text{dB}}$$

- Mayor $\omega_{BW}$ $\to$ respuesta más rápida (menor $t_r$)
- Para un sistema de 2º orden: $\omega_{BW} \approx \omega_n \sqrt{1 - 2\zeta^2 + \sqrt{4\zeta^4 - 4\zeta^2 + 2}}$

### 4.2 Pico de resonancia ($M_p$)

El **pico de resonancia** es el valor máximo de $|T(j\omega)|$ en lazo cerrado:

$$\boxed{M_p = \frac{1}{2\zeta\sqrt{1 - \zeta^2}}}$$

- Mayor $M_p$ $\to$ mayor sobreoscilación en el tiempo
- Se recomienda $M_p < 1.5$ (equivale a $\zeta > 0.4$)

### 4.3 Tabla de relaciones frecuencia-tiempo

| Especificación temporal | Especificación frecuencial | Relación |
|--------------------------|----------------------------|----------|
| Tiempo de subida $t_r$ | Ancho de banda $\omega_{BW}$ | $t_r \approx \dfrac{1.8}{\omega_{BW}}$ |
| Sobreoscilación $M_p$ | Pico de resonancia $M_r$ | Ambos dependen de $\zeta$ |
| Error en régimen permanente | Ganancia DC $|T(0)|$ | $e_{ss} = 1 - |T(0)|$ para escalón |
| Tiempo de establecimiento $t_s$ | $\omega_{BW}$ y $MF$ | Mayor $\omega_{BW}$ $\to$ menor $t_s$ |

---

## 5. Compensador en adelanto (*Lead*)

### 5.1 Función de transferencia

$$\boxed{C(s) = K_c \frac{\tau s + 1}{\alpha \tau s + 1}, \quad \alpha < 1}$$

- Tiene un **cero** en $s = -1/\tau$ y un **polo** en $s = -1/(\alpha\tau)$
- Como $\alpha < 1$, el polo está más a la izquierda que el cero
- **Efecto principal:** añade **fase positiva** en la zona de cruce de ganancia $\to$ aumenta $MF$
- **Efecto secundario:** aumenta la ganancia a altas frecuencias (amplifica ruido)

### 5.2 Máxima fase añadida

La fase máxima se produce a la frecuencia $\omega_m$:

$$\boxed{\phi_{max} = \arcsin\left(\frac{1 - \alpha}{1 + \alpha}\right)}$$

$$\boxed{\omega_m = \frac{1}{\tau\sqrt{\alpha}}}$$

### 5.3 Procedimiento de diseño (5 pasos)

**Paso 1:** Ajustar $K_c$ para cumplir la especificación de error en régimen permanente.

**Paso 2:** Con la ganancia $K_c$ fijada, obtener el Bode de $K_c G(s)$ y medir el $MF$ actual.

**Paso 3:** Calcular la fase adicional necesaria:

$$\phi_{max} = MF_{deseado} - MF_{actual} + \text{margen de seguridad} \;(5°\text{-}12°)$$

**Paso 4:** Calcular $\alpha$ a partir de $\phi_{max}$:

$$\alpha = \frac{1 - \sin(\phi_{max})}{1 + \sin(\phi_{max})}$$

**Paso 5:** Colocar $\omega_m$ en la nueva frecuencia de cruce de ganancia. La ganancia del compensador en $\omega_m$ es $1/\sqrt{\alpha}$, por lo que la nueva $\omega_{gc}$ es donde $|K_c G(j\omega)| = -10\log_{10}(1/\alpha)$ dB. Calcular $\tau = 1/(\omega_m \sqrt{\alpha})$.

In [ ]:
# Ejemplo de diseño de compensador en adelanto (lead)
# Planta: G(s) = 4 / [s(s+2)]
# Objetivo: MF >= 45° con Kv >= 20 s^-1

# Paso 1: Kv = lim s->0 s*Kc*G(s) = Kc*4/2 = 2*Kc => Kc = 10 para Kv=20
Kc_lead = 10.0
num_planta = [4.0]
den_planta = [1, 2, 0]  # s(s+2) = s^2 + 2s

# Bode de Kc*G(s) sin compensar
num_sin = [Kc_lead * 4.0]
sys_sin = signal.TransferFunction(num_sin, den_planta)
w_d = np.logspace(-1, 3, 2000)
w_s, mag_s, phase_s = signal.bode(sys_sin, w_d)

# Paso 2: MF actual
idx_gc_s = np.argmin(np.abs(mag_s))
MF_actual = 180 + phase_s[idx_gc_s]

# Paso 3: fase adicional
MF_deseado = 45.0
margen_seg = 10.0
phi_max = np.radians(MF_deseado - MF_actual + margen_seg)

# Paso 4: calcular alpha
alpha_lead = (1 - np.sin(phi_max)) / (1 + np.sin(phi_max))

# Paso 5: encontrar nueva wgc donde |Kc*G| = -10*log10(1/alpha)
ganancia_comp_wm = -10 * np.log10(1 / alpha_lead)
idx_new_gc = np.argmin(np.abs(mag_s - ganancia_comp_wm))
w_m = w_s[idx_new_gc]
tau_lead = 1 / (w_m * np.sqrt(alpha_lead))

# Construir compensador
num_comp = [Kc_lead * tau_lead, Kc_lead]
den_comp = [alpha_lead * tau_lead, 1]

# Sistema compensado
num_total = np.polymul(num_comp, num_planta)
den_total = np.polymul(den_comp, den_planta)
sys_comp = signal.TransferFunction(num_total, den_total)
w_c, mag_c, phase_c = signal.bode(sys_comp, w_d)

# MF del sistema compensado
idx_gc_c = np.argmin(np.abs(mag_c))
MF_compensado = 180 + phase_c[idx_gc_c]

# --- Gráfica: Bode antes y después ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Magnitud
ax1.semilogx(w_s, mag_s, color=COLOR_PRINCIPAL, lw=2.5, label=r'Sin compensar: $K_c G(s)$')
ax1.semilogx(w_c, mag_c, color=COLOR_COMP, lw=2.5, label=r'Compensado: $C(s)G(s)$')
ax1.axhline(0, color='gray', ls='--', lw=1)
ax1.set_ylabel(r'Magnitud (dB)')
ax1.set_title(r'Compensador en adelanto (Lead): Bode antes y después')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, which='both')

# Fase
ax2.semilogx(w_s, phase_s, color=COLOR_PRINCIPAL, lw=2.5, label=r'Sin compensar')
ax2.semilogx(w_c, phase_c, color=COLOR_COMP, lw=2.5, label=r'Compensado')
ax2.axhline(-180, color='gray', ls='--', lw=1)

# Marcar MF antes y después
ax2.annotate(f'MF = {MF_actual:.1f}\u00b0', xy=(w_s[idx_gc_s], phase_s[idx_gc_s]),
             xytext=(w_s[idx_gc_s] * 3, phase_s[idx_gc_s] + 20),
             fontsize=12, color=COLOR_PRINCIPAL, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=COLOR_PRINCIPAL, lw=2),
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_PRINCIPAL))
ax2.annotate(f'MF = {MF_compensado:.1f}\u00b0', xy=(w_c[idx_gc_c], phase_c[idx_gc_c]),
             xytext=(w_c[idx_gc_c] * 3, phase_c[idx_gc_c] + 20),
             fontsize=12, color=COLOR_COMP, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=COLOR_COMP, lw=2),
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_COMP))

ax2.set_xlabel(r'Frecuencia $\omega$ (rad/s)')
ax2.set_ylabel(r'Fase (grados)')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f'Par\u00e1metros del compensador en adelanto:')
print(f'  \u03b1 = {alpha_lead:.4f}')
print(f'  \u03c4 = {tau_lead:.4f} s')
print(f'  Kc = {Kc_lead:.1f}')
print(f'  \u03c6_max = {np.degrees(phi_max):.1f}\u00b0')
print(f'  \u03c9_m = {w_m:.2f} rad/s')
print(f'  MF antes = {MF_actual:.1f}\u00b0, MF despu\u00e9s = {MF_compensado:.1f}\u00b0')

#### Ejercicio resuelto: Diseño de compensador en adelanto

**Datos:** Planta $G(s) = \frac{4}{s(s+2)}$. Especificaciones: $MF \geq 45°$, $K_v \geq 20\;\text{s}^{-1}$.

**Paso 1:** Ajustar $K_c$ para la constante de velocidad:

$$K_v = \lim_{s \to 0} s \cdot K_c \cdot G(s) = K_c \cdot \frac{4}{2} = 2K_c$$

$$2K_c \geq 20 \implies \boxed{K_c = 10}$$

**Paso 2:** Con $K_c = 10$, se obtiene el Bode de $10G(s) = \frac{40}{s(s+2)}$. El margen de fase actual resulta ser muy pequeño (el sistema está cerca de la inestabilidad).

**Paso 3:** Fase adicional necesaria:

$$\phi_{max} = 45° - MF_{actual} + 10° \;(\text{margen de seguridad})$$

**Paso 4:** Cálculo de $\alpha$:

$$\alpha = \frac{1 - \sin(\phi_{max})}{1 + \sin(\phi_{max})}$$

**Paso 5:** Ubicar $\omega_m$ donde $|K_c G(j\omega)| = -10\log_{10}(1/\alpha)$ dB y calcular $\tau = \frac{1}{\omega_m \sqrt{\alpha}}$.

**Resultado:** El compensador en adelanto añade fase positiva en la zona de cruce, elevando el margen de fase hasta superar los $45°$ requeridos.

---

## 6. Compensador en atraso (*Lag*)

### 6.1 Función de transferencia

$$\boxed{C(s) = K_c \frac{\tau s + 1}{\beta \tau s + 1}, \quad \beta > 1}$$

- Tiene un **cero** en $s = -1/\tau$ y un **polo** en $s = -1/(\beta\tau)$
- Como $\beta > 1$, el polo está más cerca del origen que el cero
- **Efecto principal:** reduce la ganancia a **altas frecuencias** sin afectar la fase en la zona de cruce $\to$ aumenta $MG$
- **Efecto secundario:** añade fase negativa a bajas frecuencias (se coloca lejos de $\omega_{gc}$)

### 6.2 Procedimiento de diseño

**Paso 1:** Ajustar $K_c$ para cumplir la especificación de error en régimen permanente.

**Paso 2:** Obtener el Bode de $K_c G(s)$ y encontrar la frecuencia $\omega_{gc}^{nueva}$ donde la fase es $-180° + MF_{deseado} + 5°$ (margen de seguridad).

**Paso 3:** Calcular $\beta$ como la atenuación necesaria:

$$\beta = 10^{|K_c G(j\omega_{gc}^{nueva})| / 20}$$

**Paso 4:** Colocar el cero una década por debajo de $\omega_{gc}^{nueva}$:

$$\frac{1}{\tau} = \frac{\omega_{gc}^{nueva}}{10} \implies \tau = \frac{10}{\omega_{gc}^{nueva}}$$

**Paso 5:** El polo queda en $1/(\beta\tau)$.

In [ ]:
# Ejemplo de diseño de compensador en atraso (lag)
# Planta: G(s) = 1 / [s(s+1)(s+5)]
# Objetivo: MF >= 40° con Kv >= 5 s^-1

# Paso 1: Kv = lim s->0 s*Kc*G(s) = Kc/(1*5) = Kc/5 => Kc = 25 para Kv=5
Kc_lag = 25.0
num_p2 = [1.0]
den_p2 = np.polymul(np.polymul([1, 0], [1, 1]), [1, 5])  # s(s+1)(s+5)

# Bode sin compensar
num_sin2 = [Kc_lag]
sys_sin2 = signal.TransferFunction(num_sin2, den_p2)
w_d2 = np.logspace(-2, 2, 2000)
w_s2, mag_s2, phase_s2 = signal.bode(sys_sin2, w_d2)

# Paso 2: Encontrar frecuencia donde fase = -180 + 40 + 5 = -135°
target_phase = -180 + 40 + 5  # -135°
idx_new_gc = np.argmin(np.abs(phase_s2 - target_phase))
w_gc_new = w_s2[idx_new_gc]
mag_at_new_gc = mag_s2[idx_new_gc]

# Paso 3: beta = atenuación necesaria
beta_lag = 10 ** (mag_at_new_gc / 20)

# Paso 4: cero una década por debajo
tau_lag = 10 / w_gc_new

# Construir compensador lag
num_comp_lag = [Kc_lag * tau_lag, Kc_lag]
den_comp_lag = [beta_lag * tau_lag, 1]

# Sistema compensado
num_total2 = np.polymul(num_comp_lag, num_p2)
den_total2 = np.polymul(den_comp_lag, den_p2)
sys_comp2 = signal.TransferFunction(num_total2, den_total2)
w_c2, mag_c2, phase_c2 = signal.bode(sys_comp2, w_d2)

# MF del sistema compensado
idx_gc_c2 = np.argmin(np.abs(mag_c2))
MF_comp2 = 180 + phase_c2[idx_gc_c2]

# MF sin compensar
idx_gc_s2 = np.argmin(np.abs(mag_s2))
MF_sin2 = 180 + phase_s2[idx_gc_s2]

# --- Gráfica ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

ax1.semilogx(w_s2, mag_s2, color=COLOR_PRINCIPAL, lw=2.5, label=r'Sin compensar: $K_c G(s)$')
ax1.semilogx(w_c2, mag_c2, color=COLOR_COMP, lw=2.5, label=r'Compensado: $C(s)G(s)$')
ax1.axhline(0, color='gray', ls='--', lw=1)
ax1.set_ylabel(r'Magnitud (dB)')
ax1.set_title(r'Compensador en atraso (Lag): Bode antes y después')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, which='both')

ax2.semilogx(w_s2, phase_s2, color=COLOR_PRINCIPAL, lw=2.5, label=r'Sin compensar')
ax2.semilogx(w_c2, phase_c2, color=COLOR_COMP, lw=2.5, label=r'Compensado')
ax2.axhline(-180, color='gray', ls='--', lw=1)

ax2.annotate(f'MF = {MF_sin2:.1f}\u00b0', xy=(w_s2[idx_gc_s2], phase_s2[idx_gc_s2]),
             xytext=(w_s2[idx_gc_s2] * 4, phase_s2[idx_gc_s2] + 15),
             fontsize=12, color=COLOR_PRINCIPAL, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=COLOR_PRINCIPAL, lw=2),
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_PRINCIPAL))
ax2.annotate(f'MF = {MF_comp2:.1f}\u00b0', xy=(w_c2[idx_gc_c2], phase_c2[idx_gc_c2]),
             xytext=(w_c2[idx_gc_c2] * 4, phase_c2[idx_gc_c2] + 15),
             fontsize=12, color=COLOR_COMP, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=COLOR_COMP, lw=2),
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_COMP))

ax2.set_xlabel(r'Frecuencia $\omega$ (rad/s)')
ax2.set_ylabel(r'Fase (grados)')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f'Par\u00e1metros del compensador en atraso:')
print(f'  \u03b2 = {beta_lag:.2f}')
print(f'  \u03c4 = {tau_lag:.4f} s')
print(f'  Kc = {Kc_lag:.1f}')
print(f'  MF antes = {MF_sin2:.1f}\u00b0, MF despu\u00e9s = {MF_comp2:.1f}\u00b0')

#### Ejercicio resuelto: Diseño de compensador en atraso

**Datos:** Planta $G(s) = \frac{1}{s(s+1)(s+5)}$. Especificaciones: $MF \geq 40°$, $K_v \geq 5\;\text{s}^{-1}$.

**Paso 1:** Constante de velocidad:

$$K_v = \lim_{s \to 0} s \cdot K_c \cdot G(s) = K_c \cdot \frac{1}{1 \cdot 5} = \frac{K_c}{5}$$

$$\frac{K_c}{5} \geq 5 \implies \boxed{K_c = 25}$$

**Paso 2:** Con $K_c = 25$, obtener el Bode y buscar la frecuencia donde la fase es $-180° + 40° + 5° = -135°$. Esta será la nueva $\omega_{gc}$.

**Paso 3:** La atenuación necesaria en la nueva $\omega_{gc}$ es la magnitud actual en dB:

$$\beta = 10^{|K_c G(j\omega_{gc}^{nueva})|_{dB} / 20}$$

**Paso 4:** Cero del compensador una década por debajo de $\omega_{gc}^{nueva}$:

$$\tau = \frac{10}{\omega_{gc}^{nueva}}$$

**Resultado:** El compensador en atraso reduce la ganancia a altas frecuencias sin alterar significativamente la fase en la zona de cruce, logrando el $MF \geq 40°$ deseado.

---

## 7. Compensador adelanto-atraso (*Lead-Lag*)

Cuando se necesita mejorar **simultáneamente** el margen de fase y cumplir especificaciones de error en régimen permanente que requieren alta ganancia, se utiliza un compensador **adelanto-atraso**:

$$\boxed{C(s) = K_c \underbrace{\frac{\tau_1 s + 1}{\alpha \tau_1 s + 1}}_{\text{Adelanto}} \cdot \underbrace{\frac{\tau_2 s + 1}{\beta \tau_2 s + 1}}_{\text{Atraso}}}$$

con $\alpha < 1$ y $\beta > 1$.

### Procedimiento de diseño

1. **Parte de atraso:** Se diseña primero para cumplir la especificación de error permanente y reducir la ganancia a la zona deseada
2. **Parte de adelanto:** Se diseña sobre el sistema ya compensado con atraso para añadir la fase necesaria

**Ventajas:**
- La parte de **atraso** proporciona alta ganancia DC (bajo error permanente) y reduce ganancia a altas frecuencias
- La parte de **adelanto** añade fase positiva en la zona de cruce
- Se evita el aumento excesivo de ancho de banda que produce el lead solo

**Error frecuente:** Diseñar primero el adelanto y luego el atraso. El orden correcto es **atraso primero, adelanto después**, ya que el atraso modifica la ganancia que afecta al diseño del adelanto.

---

## 8. Controladores P, PI, PD, PID en el dominio frecuencial

### 8.1 Efecto de cada controlador en el Bode

| Controlador | $C(s)$ | Efecto en magnitud | Efecto en fase |
|-------------|--------|--------------------|-----------------|
| P | $K_p$ | Sube/baja uniformemente | Sin efecto |
| PI | $K_p\left(1 + \frac{1}{T_i s}\right)$ | +20 dB/dec a bajas freq | $-90°$ a baja freq, se recupera |
| PD | $K_p(1 + T_d s)$ | +20 dB/dec a altas freq | $+90°$ a alta freq |
| PID | $K_p\left(1 + \frac{1}{T_i s} + T_d s\right)$ | Combina PI + PD | Combina PI + PD |

### 8.2 Interpretación frecuencial

- **P:** Desplaza la curva de magnitud arriba/abajo. No cambia la fase $\to$ no mejora MF directamente.
- **PI:** Añade un polo en el origen (integrador) $\to$ aumenta el tipo del sistema $\to$ reduce error permanente. **Pero** añade $-90°$ de fase a bajas frecuencias, lo que puede **reducir MF**.
- **PD:** Añade un cero $\to$ añade fase positiva $\to$ **aumenta MF**. Pero amplifica ruido a altas frecuencias.
- **PID:** Combina las ventajas: integrador para error permanente + derivador para fase $\to$ mejora MF y reduce error.

In [ ]:
# Efecto de P, PI, PD, PID en el diagrama de Bode
# Planta base: G(s) = 1 / [(s+1)(s+5)]
num_base = [1.0]
den_base = np.polymul([1, 1], [1, 5])

w_pid = np.logspace(-2, 3, 2000)

# Parámetros PID
Kp = 10.0
Ti = 1.0
Td = 0.1

# Controladores
# P: C(s) = Kp
sys_P = signal.TransferFunction([Kp], den_base)
# PI: C(s) = Kp*(Ti*s + 1)/(Ti*s)
sys_PI = signal.TransferFunction(np.polymul([Kp * Ti, Kp], num_base),
                                  np.polymul([Ti, 0], den_base))
# PD: C(s) = Kp*(Td*s + 1)
sys_PD = signal.TransferFunction(np.polymul([Kp * Td, Kp], num_base),
                                  den_base)
# PID: C(s) = Kp*(Ti*Td*s^2 + Ti*s + 1)/(Ti*s)
num_pid_c = [Kp * Ti * Td, Kp * Ti, Kp]
den_pid_c = [Ti, 0]
sys_PID = signal.TransferFunction(np.polymul(num_pid_c, num_base),
                                   np.polymul(den_pid_c, den_base))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

controladores = [
    (sys_P, 'P', COLOR_PRINCIPAL, '-'),
    (sys_PI, 'PI', COLOR_RECTA, '--'),
    (sys_PD, 'PD', COLOR_PUNTO, '-.'),
    (sys_PID, 'PID', COLOR_COMP, '-'),
]

for sys_ctrl, nombre, color, estilo in controladores:
    w_out, mag, phase = signal.bode(sys_ctrl, w_pid)
    ax1.semilogx(w_out, mag, color=color, lw=2.5, ls=estilo, label=nombre)
    ax2.semilogx(w_out, phase, color=color, lw=2.5, ls=estilo, label=nombre)

ax1.axhline(0, color='gray', ls='--', lw=1)
ax1.set_ylabel(r'Magnitud (dB)')
ax1.set_title(r'Efecto de controladores P, PI, PD, PID en el Bode')
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3, which='both')

ax2.axhline(-180, color='gray', ls='--', lw=1)
ax2.set_xlabel(r'Frecuencia $\omega$ (rad/s)')
ax2.set_ylabel(r'Fase (grados)')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

---

## 9. Ejercicios resueltos

### Ejercicio 9.1: Estabilidad mediante Nyquist

**Enunciado:** Determinar la estabilidad en lazo cerrado de $G(s) = \frac{K}{(s+1)(s+3)(s+5)}$ para $K = 60$ y $K = 200$.

**Solución:**

El sistema en lazo abierto es estable ($P = 0$), por lo que necesitamos $N = 0$ para estabilidad en LC.

**Frecuencia de cruce de fase:** La fase de $G(j\omega)$ alcanza $-180°$ cuando:

$$\angle G(j\omega_{pc}) = -\arctan(\omega) - \arctan(\omega/3) - \arctan(\omega/5) = -180°$$

Resolviendo numéricamente: $\omega_{pc} \approx 5.57$ rad/s

**Magnitud en $\omega_{pc}$:**

$$|G(j\omega_{pc})| = \frac{K}{\sqrt{1+\omega_{pc}^2} \cdot \sqrt{9+\omega_{pc}^2} \cdot \sqrt{25+\omega_{pc}^2}}$$

Para $K = 60$: $|G(j\omega_{pc})| < 1$ $\to$ la curva de Nyquist **no encierra** $(-1,0)$ $\to$ $N = 0$ $\to$ $\boxed{\text{ESTABLE}}$

Para $K = 200$: $|G(j\omega_{pc})| > 1$ $\to$ la curva de Nyquist **encierra** $(-1,0)$ $\to$ $N = 1$ $\to$ $\boxed{\text{INESTABLE}}$

In [ ]:
# Nyquist para K=60 (estable) y K=200 (inestable)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
den_ej1 = np.polymul(np.polymul([1, 1], [1, 3]), [1, 5])
w_ej1 = np.logspace(-2, 2, 3000)
s_jw_ej1 = 1j * w_ej1

for ax, K, titulo, color, estab in [
    (axes[0], 60, 'K = 60 (Estable)', COLOR_PUNTO, True),
    (axes[1], 200, 'K = 200 (Inestable)', COLOR_RECTA, False),
]:
    G_ej1 = K * np.polyval([1.0], s_jw_ej1) / np.polyval(den_ej1, s_jw_ej1)
    ax.plot(G_ej1.real, G_ej1.imag, color=color, lw=2.5, label=r'$\omega > 0$')
    ax.plot(G_ej1.real, -G_ej1.imag, color=color, lw=2.5, ls='--', alpha=0.5, label=r'$\omega < 0$')
    ax.plot(-1, 0, 'x', color='black', ms=14, mew=3, zorder=5)
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    ax.set_xlabel(r'Parte Real')
    ax.set_ylabel(r'Parte Imaginaria')
    ax.set_title(titulo)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    if estab:
        ax.set_xlim(-1.5, 0.5)
        ax.set_ylim(-1.5, 1.5)
        ax.annotate(r'$(-1,0)$ NO encerrado' + '\n' + r'$\to$ Estable',
                    xy=(-1, 0), xytext=(-0.2, 1.0),
                    fontsize=11, color=COLOR_PUNTO, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2),
                    bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_PUNTO))
    else:
        ax.set_xlim(-4, 2)
        ax.set_ylim(-4, 4)
        ax.annotate(r'$(-1,0)$ ENCERRADO' + '\n' + r'$\to$ Inestable',
                    xy=(-1, 0), xytext=(0.5, 2.5),
                    fontsize=11, color=COLOR_RECTA, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color=COLOR_RECTA, lw=2),
                    bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_RECTA))

plt.tight_layout()
plt.show()

### Ejercicio 9.2: Cálculo de márgenes desde Bode

**Enunciado:** Para $G(s) = \frac{50}{(s+2)(s+5)(s+10)}$, calcular $MG$ y $MF$.

**Solución:**

**Frecuencia de cruce de ganancia ($\omega_{gc}$):** Se busca $|G(j\omega_{gc})| = 1$ (0 dB).

$$|G(j\omega)| = \frac{50}{\sqrt{4 + \omega^2} \cdot \sqrt{25 + \omega^2} \cdot \sqrt{100 + \omega^2}} = 1$$

Resolviendo numéricamente: $\omega_{gc} \approx 1.65$ rad/s

**Fase en $\omega_{gc}$:**

$$\angle G(j\omega_{gc}) = -\arctan\left(\frac{\omega_{gc}}{2}\right) - \arctan\left(\frac{\omega_{gc}}{5}\right) - \arctan\left(\frac{\omega_{gc}}{10}\right)$$

$$\angle G(j1.65) \approx -39.5° - 18.3° - 9.4° = -67.2°$$

$$\boxed{MF = 180° + (-67.2°) = 112.8°}$$

**Frecuencia de cruce de fase ($\omega_{pc}$):** Se busca $\angle G(j\omega_{pc}) = -180°$.

Resolviendo: $\omega_{pc} \approx 12.25$ rad/s

$$|G(j\omega_{pc})| \approx 0.032 \implies |G|_{dB} = 20\log_{10}(0.032) \approx -29.9\;\text{dB}$$

$$\boxed{MG = -(-29.9) = 29.9\;\text{dB}}$$

Ambos márgenes son amplios $\to$ el sistema es **muy estable** en lazo cerrado.

### Ejercicio 9.3: Diseño completo de compensador en adelanto

**Enunciado:** Para la planta $G(s) = \frac{10}{s(s+2)(s+8)}$, diseñar un compensador en adelanto tal que $MF \geq 50°$ y $K_v \geq 10\;\text{s}^{-1}$.

**Paso 1:** Constante de velocidad:

$$K_v = \lim_{s \to 0} s \cdot K_c \cdot \frac{10}{s(s+2)(s+8)} = K_c \cdot \frac{10}{16} = \frac{5K_c}{8}$$

$$\frac{5K_c}{8} \geq 10 \implies K_c \geq 16 \implies \boxed{K_c = 16}$$

**Paso 2:** Obtener el Bode de $16G(s) = \frac{160}{s(s+2)(s+8)}$ y medir el margen de fase actual.

**Paso 3:** Si $MF_{actual} \approx 10°$, la fase adicional necesaria es:

$$\phi_{max} = 50° - 10° + 10° = 50°$$

**Paso 4:**

$$\alpha = \frac{1 - \sin(50°)}{1 + \sin(50°)} = \frac{1 - 0.766}{1 + 0.766} = \frac{0.234}{1.766} = 0.1325$$

**Paso 5:** Buscar $\omega_m$ donde $|K_c G(j\omega)| = -10\log_{10}(1/0.1325) = -8.78$ dB. Calcular $\tau = 1/(\omega_m \sqrt{0.1325})$.

$$\boxed{C(s) = 16 \cdot \frac{\tau s + 1}{0.1325\tau s + 1}}$$

In [ ]:
# Verificación del ejercicio 9.3
Kc_93 = 16.0
num_93 = [10.0]
den_93 = np.polymul(np.polymul([1, 0], [1, 2]), [1, 8])

# Sin compensar
sys_93s = signal.TransferFunction([Kc_93 * 10.0], den_93)
w_93 = np.logspace(-2, 3, 2000)
w93, mag93, phase93 = signal.bode(sys_93s, w_93)

idx_gc93 = np.argmin(np.abs(mag93))
MF_93_sin = 180 + phase93[idx_gc93]

# Diseño del compensador lead
phi_max_93 = np.radians(50 - MF_93_sin + 10)
if phi_max_93 > np.radians(70):
    phi_max_93 = np.radians(65)  # limitar
alpha_93 = (1 - np.sin(phi_max_93)) / (1 + np.sin(phi_max_93))
gan_comp_93 = -10 * np.log10(1 / alpha_93)
idx_wm93 = np.argmin(np.abs(mag93 - gan_comp_93))
wm_93 = w93[idx_wm93]
tau_93 = 1 / (wm_93 * np.sqrt(alpha_93))

# Sistema compensado
num_c93 = np.polymul([Kc_93 * tau_93, Kc_93], num_93)
den_c93 = np.polymul([alpha_93 * tau_93, 1], den_93)
sys_93c = signal.TransferFunction(num_c93, den_c93)
w93c, mag93c, phase93c = signal.bode(sys_93c, w_93)

idx_gc93c = np.argmin(np.abs(mag93c))
MF_93_comp = 180 + phase93c[idx_gc93c]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

ax1.semilogx(w93, mag93, color=COLOR_PRINCIPAL, lw=2.5, label=r'Sin compensar')
ax1.semilogx(w93c, mag93c, color=COLOR_COMP, lw=2.5, label=r'Con lead')
ax1.axhline(0, color='gray', ls='--', lw=1)
ax1.set_ylabel(r'Magnitud (dB)')
ax1.set_title(r'Ejercicio 9.3: $G(s) = \frac{10}{s(s+2)(s+8)}$ con compensador lead')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, which='both')

ax2.semilogx(w93, phase93, color=COLOR_PRINCIPAL, lw=2.5, label=f'Sin compensar (MF={MF_93_sin:.1f}\u00b0)')
ax2.semilogx(w93c, phase93c, color=COLOR_COMP, lw=2.5, label=f'Con lead (MF={MF_93_comp:.1f}\u00b0)')
ax2.axhline(-180, color='gray', ls='--', lw=1)
ax2.set_xlabel(r'Frecuencia $\omega$ (rad/s)')
ax2.set_ylabel(r'Fase (grados)')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f'MF sin compensar: {MF_93_sin:.1f}\u00b0')
print(f'MF con lead: {MF_93_comp:.1f}\u00b0')
print(f'\u03b1 = {alpha_93:.4f}, \u03c4 = {tau_93:.4f} s')

### Ejercicio 9.4: Diseño de compensador en atraso

**Enunciado:** Para $G(s) = \frac{1}{s(s+1)(s+2)}$, diseñar un compensador en atraso con $MF \geq 40°$ y $K_v \geq 5\;\text{s}^{-1}$.

**Paso 1:**

$$K_v = \lim_{s \to 0} s \cdot K_c \cdot \frac{1}{s(s+1)(s+2)} = \frac{K_c}{2}$$

$$\frac{K_c}{2} \geq 5 \implies \boxed{K_c = 10}$$

**Paso 2:** Bode de $10G(s) = \frac{10}{s(s+1)(s+2)}$. Buscar $\omega$ donde fase $= -180° + 40° + 5° = -135°$.

**Paso 3:** En esa frecuencia, la magnitud es positiva (en dB), por lo que necesitamos atenuar:

$$\beta = 10^{\text{mag}_{dB}/20}$$

**Paso 4:** Cero una década por debajo: $\tau = 10/\omega_{gc}^{nueva}$

$$\boxed{C(s) = 10 \cdot \frac{\tau s + 1}{\beta \tau s + 1}}$$

### Ejercicio 9.5: Análisis frecuencial completo

**Enunciado:** Para $G(s) = \frac{100}{(s+1)(s+10)}$ en lazo cerrado unitario, determinar:

a) $MG$ y $MF$

b) $\omega_{BW}$ y $M_p$ en lazo cerrado

c) Estabilidad mediante Nyquist

**Solución parte a):**

$$|G(j\omega)| = \frac{100}{\sqrt{1 + \omega^2} \cdot \sqrt{100 + \omega^2}}$$

Para $|G(j\omega_{gc})| = 1$:

$(1 + \omega^2)(100 + \omega^2) = 10000$

$\omega^4 + 101\omega^2 + 100 = 10000$

$\omega^4 + 101\omega^2 - 9900 = 0$

Resolviendo: $\omega^2 = \frac{-101 + \sqrt{101^2 + 4 \cdot 9900}}{2} = \frac{-101 + \sqrt{49801}}{2} = \frac{-101 + 223.2}{2} = 61.1$

$\omega_{gc} \approx 7.82$ rad/s

$$\angle G(j7.82) = -\arctan(7.82) - \arctan(7.82/10) = -82.7° - 38.0° = -120.7°$$

$$\boxed{MF = 180° - 120.7° = 59.3°}$$

Para la fase de cruce ($-180°$): con solo 2 polos (sin integradores) la fase máxima negativa es $-180°$ (asintótica), por lo que $\omega_{pc} \to \infty$ y $|G(j\omega_{pc})| \to 0$.

$$\boxed{MG = \infty}$$

**Solución parte b):** El sistema en lazo cerrado tiene polos dominantes de segundo orden. Con $MF \approx 59°$, se estima $\zeta \approx 0.59$. Esto da $M_p \approx 1/(2 \cdot 0.59 \cdot \sqrt{1-0.59^2}) \approx 1.05$.

**Solución parte c):** Con solo 2 polos en el SPD izquierdo, la curva de Nyquist no puede encerrar $(-1,0)$ $\to$ $\boxed{\text{ESTABLE}}$ para cualquier ganancia positiva.

### Ejercicio 9.6: Diseño frecuencial completo con verificación

**Enunciado:** Para $G(s) = \frac{5}{s(s+1)(s+5)}$, diseñar un controlador para que $MF \geq 45°$, $K_v \geq 30\;\text{s}^{-1}$ y $\omega_{BW} \geq 5$ rad/s.

**Análisis:** Se necesita alta ganancia ($K_v = 30$) y alto ancho de banda, lo que sugiere un compensador **adelanto-atraso** (lead-lag).

**Paso 1 (Ganancia):**

$$K_v = K_c \cdot \frac{5}{1 \cdot 5} = K_c \implies K_c = 30$$

**Paso 2 (Parte lag):** Con $K_c = 30$, el Bode muestra un MF muy pequeño. Se busca la frecuencia donde la fase es aproximadamente $-180° + 45° + 5° + 10° = -120°$ (el extra de $10°$ compensa la fase negativa residual del lag). Se calcula $\beta$ y $\tau_2$ según el procedimiento del lag.

**Paso 3 (Parte lead):** Sobre el sistema compensado con lag, se calcula la fase adicional necesaria y se diseña el lead con $\alpha$ y $\tau_1$.

$$\boxed{C(s) = 30 \cdot \frac{\tau_1 s + 1}{\alpha \tau_1 s + 1} \cdot \frac{\tau_2 s + 1}{\beta \tau_2 s + 1}}$$

In [ ]:
# Verificación del ejercicio 9.6: Lead-Lag
Kc_96 = 30.0
num_96 = [5.0]
den_96 = np.polymul(np.polymul([1, 0], [1, 1]), [1, 5])

# Sin compensar
sys_96s = signal.TransferFunction([Kc_96 * 5.0], den_96)
w_96 = np.logspace(-3, 3, 3000)
w96, mag96, phase96 = signal.bode(sys_96s, w_96)

# Lag: buscar fase = -120°
target_ph_96 = -120
idx_lag96 = np.argmin(np.abs(phase96 - target_ph_96))
w_gc_lag96 = w96[idx_lag96]
mag_lag96 = mag96[idx_lag96]
beta_96 = 10 ** (mag_lag96 / 20)
tau2_96 = 10 / w_gc_lag96

# Sistema con lag
num_lag96 = np.polymul([Kc_96 * tau2_96, Kc_96], num_96)
den_lag96 = np.polymul([beta_96 * tau2_96, 1], den_96)
sys_96lag = signal.TransferFunction(num_lag96, den_lag96)
w96l, mag96l, phase96l = signal.bode(sys_96lag, w_96)

# Lead sobre el sistema con lag
idx_gc_lag96 = np.argmin(np.abs(mag96l))
MF_lag96 = 180 + phase96l[idx_gc_lag96]
phi_lead96 = np.radians(max(45 - MF_lag96 + 10, 5))
alpha_96 = (1 - np.sin(phi_lead96)) / (1 + np.sin(phi_lead96))
alpha_96 = max(alpha_96, 0.05)  # limitar
gan_lead96 = -10 * np.log10(1 / alpha_96)
idx_wm96 = np.argmin(np.abs(mag96l - gan_lead96))
wm_96 = w96l[idx_wm96]
tau1_96 = 1 / (wm_96 * np.sqrt(alpha_96))

# Sistema final con lead-lag
num_final96 = np.polymul([tau1_96, 1], num_lag96)
den_final96 = np.polymul([alpha_96 * tau1_96, 1], den_lag96)
sys_96f = signal.TransferFunction(num_final96, den_final96)
w96f, mag96f, phase96f = signal.bode(sys_96f, w_96)

idx_gc96f = np.argmin(np.abs(mag96f))
MF_96f = 180 + phase96f[idx_gc96f]

idx_gc96s = np.argmin(np.abs(mag96))
MF_96s = 180 + phase96[idx_gc96s]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

ax1.semilogx(w96, mag96, color=COLOR_PRINCIPAL, lw=2, label=f'Sin compensar')
ax1.semilogx(w96l, mag96l, color=COLOR_FASE, lw=2, ls='--', label='Con lag')
ax1.semilogx(w96f, mag96f, color=COLOR_COMP, lw=2.5, label='Con lead-lag')
ax1.axhline(0, color='gray', ls='--', lw=1)
ax1.set_ylabel(r'Magnitud (dB)')
ax1.set_title(r'Ejercicio 9.6: Dise\u00f1o Lead-Lag para $G(s) = \frac{5}{s(s+1)(s+5)}$')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, which='both')

ax2.semilogx(w96, phase96, color=COLOR_PRINCIPAL, lw=2, label=f'Sin compensar (MF={MF_96s:.1f}\u00b0)')
ax2.semilogx(w96l, phase96l, color=COLOR_FASE, lw=2, ls='--', label=f'Con lag (MF={MF_lag96:.1f}\u00b0)')
ax2.semilogx(w96f, phase96f, color=COLOR_COMP, lw=2.5, label=f'Con lead-lag (MF={MF_96f:.1f}\u00b0)')
ax2.axhline(-180, color='gray', ls='--', lw=1)
ax2.set_xlabel(r'Frecuencia $\omega$ (rad/s)')
ax2.set_ylabel(r'Fase (grados)')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f'MF sin compensar: {MF_96s:.1f}\u00b0')
print(f'MF con lag: {MF_lag96:.1f}\u00b0')
print(f'MF con lead-lag: {MF_96f:.1f}\u00b0')

---

## 10. Catálogo completo de ejercicios: todos los patrones

| # | Tipo | Herramientas | Ecuación/criterio clave | Dificultad |
|---|------|-------------|-------------------------|------------|
| 1 | Estabilidad por Nyquist (fase mínima) | Nyquist, conteo de encirculamientos | $Z = N + P$ | Media |
| 2 | Estabilidad por Nyquist (con polos en SPD) | Nyquist completo | $N = -P$ para estabilidad | Alta |
| 3 | Cálculo de $MG$ y $MF$ desde Bode | Bode, $\omega_{gc}$, $\omega_{pc}$ | $MF = 180° + \angle G(j\omega_{gc})$ | Baja |
| 4 | Ganancia crítica desde Bode | Bode, $\omega_{pc}$ | $K_{cr} = 1/|G(j\omega_{pc})|$ | Media |
| 5 | Diseño de compensador lead | 5 pasos lead | $\alpha = \frac{1-\sin\phi}{1+\sin\phi}$ | Alta |
| 6 | Diseño de compensador lag | 5 pasos lag | $\beta = 10^{|G|_{dB}/20}$ | Alta |
| 7 | Diseño lead-lag | Combinar lag + lead | Ambos procedimientos | Muy alta |
| 8 | Relación $MF$-$\zeta$-sobreoscilación | Bode + especificaciones temporales | $MF \approx 100\zeta$ | Media |
| 9 | Efecto de controlador P en $MG$/$MF$ | Bode con $K$ variable | Solo desplaza magnitud | Baja |
| 10 | Efecto de PI: error permanente vs $MF$ | Bode con integrador | Añade $-90°$ a baja freq | Media |
| 11 | Efecto de PD: mejora de $MF$ | Bode con derivador | Añade fase positiva | Media |
| 12 | Análisis completo: Bode + Nyquist + diseño | Todas las herramientas | Combinar criterios | Muy alta |

### 10.1 Tipo 1: Estabilidad por Nyquist (fase mínima)

$$\boxed{\text{Si } P = 0 \text{ y la curva NO encierra } (-1,0) \implies \text{Estable en LC}}$$

**Procedimiento:**
1. Verificar que $G_{BA}(s)$ no tiene polos en el SPD ($P = 0$)
2. Evaluar $G_{BA}(j\omega)$ para $\omega \in [0, \infty)$
3. Trazar el diagrama de Nyquist
4. Observar si $(-1, 0)$ queda encerrado

**Cómo afecta la ganancia:**
- Si $K$ **aumenta** $\to$ la curva de Nyquist se **escala** hacia afuera $\to$ más probable encerrar $(-1,0)$ $\to$ más probable inestabilidad
- Si $K$ **disminuye** $\to$ la curva se contrae $\to$ más estable

### 10.2 Tipo 2: Estabilidad por Nyquist (polos en SPD)

$$\boxed{Z = N + P, \quad \text{necesitamos } N = -P \text{ (encirculamientos antihorarios)}}$$

**Procedimiento:**
1. Contar los polos de $G_{BA}(s)$ en el SPD: $P$
2. Trazar Nyquist completo (incluyendo cierre al infinito)
3. Contar encirculamientos $N$ (horario = $+1$, antihorario = $-1$)
4. Calcular $Z = N + P$
5. Si $Z = 0$ $\to$ estable. Si $Z > 0$ $\to$ inestable con $Z$ polos en SPD

**Error frecuente:** Olvidar que para sistemas con polos en el eje imaginario (integradores), se necesita una indentación (*indentation*) del contorno de Nyquist.

### 10.3 Tipo 3: Cálculo de MG y MF desde Bode

$$\boxed{MF = 180° + \angle G(j\omega_{gc})} \qquad \boxed{MG = -20\log_{10}|G(j\omega_{pc})|}$$

**Procedimiento:**
1. Trazar el Bode de magnitud y fase de $G(j\omega)$
2. En el Bode de magnitud, encontrar $\omega_{gc}$ donde $|G| = 0$ dB
3. Leer la fase en $\omega_{gc}$ $\to$ calcular $MF$
4. En el Bode de fase, encontrar $\omega_{pc}$ donde fase $= -180°$
5. Leer la magnitud en $\omega_{pc}$ $\to$ calcular $MG$

**Truco para el examen:** Si $\omega_{gc} < \omega_{pc}$ y ambos márgenes son positivos $\to$ el sistema es estable.

### 10.4 Tipo 4: Ganancia crítica

$$\boxed{K_{cr} = \frac{1}{|G(j\omega_{pc})|}}$$

La **ganancia crítica** es el valor de $K$ que hace $MG = 0$ dB, es decir, el límite de estabilidad.

**Procedimiento:**
1. Encontrar $\omega_{pc}$ donde $\angle G(j\omega_{pc}) = -180°$
2. Calcular $|G(j\omega_{pc})|$
3. $K_{cr} = 1 / |G(j\omega_{pc})|$
4. Para $K < K_{cr}$: estable. Para $K > K_{cr}$: inestable

### 10.5 Tipo 5: Diseño de compensador en adelanto (lead)

$$\boxed{C(s) = K_c \frac{\tau s + 1}{\alpha \tau s + 1}, \quad \alpha < 1}$$

**Procedimiento completo (5 pasos):**

1. $K_c$ desde especificación de error permanente
2. Bode de $K_c G(s)$ $\to$ medir $MF_{actual}$
3. $\phi_{max} = MF_{deseado} - MF_{actual} + \text{margen}$ (5°-12°)
4. $\alpha = \dfrac{1 - \sin(\phi_{max})}{1 + \sin(\phi_{max})}$
5. $\omega_m$ donde $|K_c G| = -10\log(1/\alpha)$ dB, $\tau = \dfrac{1}{\omega_m \sqrt{\alpha}}$

**Cómo afectan los parámetros:**
- Si $\alpha$ **disminuye** $\to$ más fase añadida $\to$ pero más amplificación de ruido
- Si $\tau$ **aumenta** $\to$ el compensador actúa a frecuencias más bajas
- Límite práctico: $\phi_{max} \leq 65°$ con un solo lead. Para más, usar dos en cascada

### 10.6 Tipo 6: Diseño de compensador en atraso (lag)

$$\boxed{C(s) = K_c \frac{\tau s + 1}{\beta \tau s + 1}, \quad \beta > 1}$$

**Procedimiento:**

1. $K_c$ desde error permanente
2. Buscar nueva $\omega_{gc}$ donde fase $= -180° + MF_{deseado} + 5°$
3. $\beta = 10^{|K_c G(j\omega_{gc}^{nueva})|_{dB}/20}$
4. $\tau = 10 / \omega_{gc}^{nueva}$ (cero una década por debajo)
5. Polo en $1/(\beta \tau)$

**Cómo afectan los parámetros:**
- Si $\beta$ **aumenta** $\to$ más atenuación a altas frecuencias $\to$ pero respuesta más lenta
- El compensador lag **reduce** el ancho de banda (respuesta más lenta)

### 10.7 Tipo 7: Diseño lead-lag

$$\boxed{C(s) = K_c \cdot \frac{\tau_1 s + 1}{\alpha \tau_1 s + 1} \cdot \frac{\tau_2 s + 1}{\beta \tau_2 s + 1}}$$

**Procedimiento:**
1. Fijar $K_c$ por error permanente
2. Diseñar parte lag (atenuar ganancia)
3. Diseñar parte lead sobre el sistema ya compensado con lag (añadir fase)
4. Verificar que el sistema final cumple todas las especificaciones

**Cuándo usar lead-lag:**
- Cuando el lead solo necesitaría $\phi_{max} > 65°$ (impracticable)
- Cuando se necesita alta ganancia DC + buen margen de fase
- Cuando se quiere limitar el ancho de banda (el lag lo reduce)

### 10.8 Tipo 8: Relación MF-amortiguamiento-sobreoscilación

$$\boxed{MF \approx 100\zeta \;\text{(para } \zeta < 0.7\text{)}}$$

$$\boxed{\%OS = 100 \cdot e^{-\pi\zeta/\sqrt{1-\zeta^2}}}$$

**Procedimiento:**
1. Desde la especificación de sobreoscilación, calcular $\zeta$
2. Desde $\zeta$, estimar $MF_{deseado} \approx 100\zeta$
3. Usar el $MF_{deseado}$ como especificación para el diseño frecuencial

| $MF$ | $\zeta$ aprox. | $\%OS$ aprox. |
|------|----------------|----------------|
| 30° | 0.30 | 37% |
| 45° | 0.45 | 21% |
| 60° | 0.60 | 10% |
| 70° | 0.70 | 5% |

### 10.9 Tipo 9: Efecto de controlador P en márgenes

$$C(s) = K_p$$

**Efecto en Bode:**
- Magnitud: se desplaza $20\log_{10}(K_p)$ dB uniformemente
- Fase: **sin cambio**

**Consecuencia:**
- Si $K_p$ aumenta $\to$ $\omega_{gc}$ se desplaza a la derecha $\to$ la fase en la nueva $\omega_{gc}$ es más negativa $\to$ **$MF$ disminuye**
- Si $K_p$ aumenta $\to$ $|G(j\omega_{pc})|$ aumenta $\to$ **$MG$ disminuye**
- **Compromiso:** aumentar $K_p$ reduce el error permanente pero empeora la estabilidad

### 10.10 Tipo 10: Efecto de controlador PI

$$C(s) = K_p \left(1 + \frac{1}{T_i s}\right) = K_p \frac{T_i s + 1}{T_i s}$$

**Efecto en Bode:**
- Añade un polo en el origen $\to$ +20 dB/dec a bajas frecuencias
- Añade un cero en $s = -1/T_i$ $\to$ la pendiente vuelve a la original para $\omega > 1/T_i$
- Fase: $-90°$ a $\omega \to 0$, se recupera a $0°$ pasado el cero

**Consecuencia:**
- **Elimina error permanente** al escalón (aumenta el tipo del sistema)
- **Reduce MF** por la fase negativa añadida
- Se coloca $1/T_i$ bien por debajo de $\omega_{gc}$ para minimizar la pérdida de MF

### 10.11 Tipo 11: Efecto de controlador PD

$$C(s) = K_p(1 + T_d s)$$

**Efecto en Bode:**
- Añade un cero en $s = -1/T_d$ $\to$ +20 dB/dec a altas frecuencias
- Fase: $0°$ a baja freq, sube hasta $+90°$ a alta freq

**Consecuencia:**
- **Aumenta MF** por la fase positiva
- **Amplifica ruido** a altas frecuencias (la magnitud crece indefinidamente)
- En la práctica se añade un polo de alta frecuencia: $C(s) = K_p \dfrac{T_d s + 1}{T_d s/N + 1}$ con $N \approx 10$

### 10.12 Tipo 12: Análisis completo (Bode + Nyquist + diseño)

Este es el tipo de problema más completo y habitual en exámenes:

**Procedimiento:**
1. **Trazar Bode** de $G(j\omega)$ (o $K \cdot G(j\omega)$ si se da la ganancia)
2. **Calcular** $MG$ y $MF$ $\to$ determinar estabilidad actual
3. **Trazar Nyquist** y verificar consistencia con Bode
4. **Comparar** márgenes con especificaciones
5. **Diseñar compensador** (lead, lag, o lead-lag según necesidad)
6. **Verificar** que el sistema compensado cumple todas las especificaciones
7. **Comprobar** la respuesta temporal (opcional, para confirmar)

**Decisión de qué compensador usar:**
- Solo falta $MF$ $\to$ **lead**
- Solo falta ganancia DC $\to$ **lag**
- Falta ambos $\to$ **lead-lag**
- Error permanente inaceptable $\to$ **PI** o **lag**

---

## 11. Tablas resumen

### Tabla 1: Fórmulas clave

| Fórmula | Uso |
|---------|-----|
| $Z = N + P$ | Criterio de Nyquist |
| $MF = 180° + \angle G(j\omega_{gc})$ | Margen de fase |
| $MG = -20\log_{10}|G(j\omega_{pc})|$ | Margen de ganancia |
| $\alpha = \dfrac{1 - \sin\phi_{max}}{1 + \sin\phi_{max}}$ | Parámetro $\alpha$ del lead |
| $\phi_{max} = \arcsin\dfrac{1-\alpha}{1+\alpha}$ | Fase máxima del lead |
| $\omega_m = \dfrac{1}{\tau\sqrt{\alpha}}$ | Frecuencia de máxima fase (lead) |
| $\beta = 10^{|G|_{dB}/20}$ | Parámetro $\beta$ del lag |
| $MF \approx 100\zeta$ | Relación margen de fase - amortiguamiento |
| $K_{cr} = 1/|G(j\omega_{pc})|$ | Ganancia crítica |

### Tabla 2: Comparación de compensadores

| Compensador | Mejora $MF$ | Mejora $MG$ | Error perm. | Ancho de banda | Ruido |
|-------------|-------------|-------------|-------------|----------------|-------|
| Lead ($\alpha < 1$) | **Sí** | No directamente | No cambia | Aumenta | Empeora |
| Lag ($\beta > 1$) | No directamente | **Sí** | Mejora | Disminuye | Mejora |
| Lead-Lag | **Sí** | **Sí** | Mejora | Controlable | Moderado |
| P | No | No | Mejora con $K$ | Aumenta con $K$ | Neutro |
| PI | Reduce | Puede reducir | **Elimina** | Disminuye | Mejora |
| PD | **Sí** | Puede mejorar | No cambia | Aumenta | Empeora |
| PID | **Sí** | Puede mejorar | **Elimina** | Aumenta | Moderado |

### Tabla 3: Valores típicos de referencia

| Parámetro | Valor recomendado | Notas |
|-----------|-------------------|-------|
| $MG$ | $> 6$ dB (ideal $> 12$ dB) | Robustez frente a variaciones de ganancia |
| $MF$ | $30° - 60°$ (ideal $\approx 45°$) | Buen compromiso rapidez/amortiguamiento |
| $\alpha$ (lead) | $0.05 - 0.5$ | Menor $\alpha$ = más fase pero más ruido |
| $\beta$ (lag) | $2 - 20$ | Mayor $\beta$ = más atenuación pero más lento |
| Margen de seguridad (lead) | $5° - 12°$ | Compensa desplazamiento de $\omega_{gc}$ |
| Cero del lag | 1 década bajo $\omega_{gc}^{nueva}$ | Minimiza efecto de fase negativa |

In [ ]:
# Gráfica resumen: efecto de la ganancia K en el Nyquist
fig, ax = plt.subplots(figsize=(10, 9))

den_resumen = np.polymul(np.polymul([1, 1], [1, 3]), [1, 5])
w_res = np.logspace(-2, 2, 2000)
s_jw_res = 1j * w_res

K_vals = [15, 60, 120]
colores_K = [COLOR_PUNTO, COLOR_PRINCIPAL, COLOR_RECTA]
estilos_K = ['-', '-', '-']

for K, col, est in zip(K_vals, colores_K, estilos_K):
    G_res = K / np.polyval(den_resumen, s_jw_res)
    ax.plot(G_res.real, G_res.imag, color=col, lw=2.5, ls=est, label=f'K = {K}')
    ax.plot(G_res.real, -G_res.imag, color=col, lw=2.5, ls='--', alpha=0.4)

ax.plot(-1, 0, 'x', color='black', ms=14, mew=3, zorder=5, label=r'$(-1, 0)$')

# Círculo unitario
theta_c = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta_c), np.sin(theta_c), 'k--', lw=0.8, alpha=0.2)

ax.set_xlabel(r'Parte Real')
ax.set_ylabel(r'Parte Imaginaria')
ax.set_title(r'Efecto de la ganancia $K$ en el diagrama de Nyquist')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlim(-2.5, 1.5)
ax.set_ylim(-2.5, 2.5)

ax.annotate('Estable', xy=(-0.3, 0.3), fontsize=14, color=COLOR_PUNTO,
            fontweight='bold')
ax.annotate('L\u00edmite', xy=(-0.8, -1.5), fontsize=14, color=COLOR_PRINCIPAL,
            fontweight='bold')
ax.annotate('Inestable', xy=(-2.0, 1.5), fontsize=14, color=COLOR_RECTA,
            fontweight='bold')

plt.tight_layout()
plt.show()